In [ ]:
# !pip install transformers

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
# model_id = "defog/sqlcoder-7b-2"  # https://huggingface.co/defog/sqlcoder-7b-2
model_id = "Yuk050/gemma-3-1b-text-to-sql-model"   # https://huggingface.co/Yuk050/gemma-3-1b-text-to-sql-model
# model_id = "prem-research/prem-1B-SQL"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

In [ ]:
PROMPT_TEMPLATE = """
### Instructions:

Your task is to convert a question into a SQL query, given a Postgres database schema.
Adhere to these rules:
- Deliberately go through the question and database schema word by word** to appropriately answer the question
- Use Table Aliases to prevent ambiguity. For example, `SELECT table1.col1, table2.col1 FROM table1 JOIN table2 ON table1.id = table2.id`.
- When creating a ratio, always cast the numerator as float

### Input:

Generate a SQL query that answers the question: `{user_query}`.

This query will run on a database whose schema is represented in this string:

CREATE TABLE products (
  product_id INTEGER PRIMARY KEY, -- Unique ID for each product
  name VARCHAR(50), -- Name of the product
  price DECIMAL(10,2), -- Price of each unit of the product
  quantity INTEGER  -- Current quantity in stock
);

CREATE TABLE customers (
   customer_id INTEGER PRIMARY KEY, -- Unique ID for each customer
   name VARCHAR(50), -- Name of the customer
   address VARCHAR(100) -- Mailing address of the customer
);

CREATE TABLE salespeople (
  salesperson_id INTEGER PRIMARY KEY, -- Unique ID for each salesperson
  name VARCHAR(50), -- Name of the salesperson
  region VARCHAR(50) -- Geographic sales region
);

CREATE TABLE sales (
  sale_id INTEGER PRIMARY KEY, -- Unique ID for each sale
  product_id INTEGER, -- ID of product sold
  customer_id INTEGER,  -- ID of customer who made purchase
  salesperson_id INTEGER, -- ID of salesperson who made the sale
  sale_date DATE, -- Date the sale occurred
  quantity INTEGER -- Quantity of product sold
);

CREATE TABLE product_suppliers (
  supplier_id INTEGER PRIMARY KEY, -- Unique ID for each supplier
  product_id INTEGER, -- Product ID supplied
  supply_price DECIMAL(10,2) -- Unit price charged by supplier
);

-- sales.product_id can be joined with products.product_id
-- sales.customer_id can be joined with customers.customer_id
-- sales.salesperson_id can be joined with salespeople.salesperson_id
-- product_suppliers.product_id can be joined with products.product_id

### Response:
Based on your instructions, here is the SQL query I have generated to answer the question: `{user_query}`:
"""

In [ ]:
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256)

In [ ]:
# query = "What are the names of the sales people? Sort the names alphabetically."
query = "What were the top 5 selling products last month by revenue?"

In [ ]:
prompt = PROMPT_TEMPLATE.format(user_query=query)

In [ ]:
result = generator(prompt, do_sample=False, return_full_text=False)[0]["generated_text"]

In [ ]:
print(">>>", query)
print("-" * 100)
print(result.replace("Ġ", " "))

<hr/>